<a href="https://colab.research.google.com/github/mohamedlgrindi/mlops-llm-platform/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install vllm torch fastapi uvicorn pyngrok nest_asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 821.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 M

In [5]:
import os
import sys
import asyncio
import nest_asyncio
import ipykernel.iostream

# 1. Patch Colab's ipykernel stream to return standard Linux stdout FD (1)
ipykernel.iostream.OutStream.fileno = lambda self: 1

# 2. Force vLLM engine settings
os.environ["VLLM_USE_V1"] = "0"

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI()

# 3. Configure engine args optimized for Colab T4 GPU
engine_args = AsyncEngineArgs(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    gpu_memory_utilization=0.75,
    max_model_len=1024,
    enforce_eager=True  # Bypasses CUDA graph capture crashes on T4 GPUs
)
engine = AsyncLLMEngine.from_engine_args(engine_args)

class GenerateReq(BaseModel):
    prompt: str
    max_tokens: int = 128
    temperature: float = 0.7

@app.post("/remote-generate")
async def generate(req: GenerateReq):
    sampling_params = SamplingParams(
        temperature=req.temperature,
        max_tokens=req.max_tokens
    )
    request_id = f"colab-{asyncio.get_event_loop().time()}"
    results_generator = engine.generate(req.prompt, sampling_params, request_id)

    async def token_stream():
        previous_text = ""
        async for request_output in results_generator:
            text = request_output.outputs[0].text
            new_text = text[len(previous_text):]
            previous_text = text
            yield f"data: {new_text}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(token_stream(), media_type="text/event-stream")

# 4. Connect ngrok tunnel (Replace with your token from dashboard.ngrok.com)
NGROK_TOKEN = "3D0QjXb3z5iZRLU4NUe3KIqZai2_2YVGwrNybKRV3x8XWepcs"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000)
print(f"🚀 Real GPU vLLM Server Live at: {public_url}/remote-generate")

# 5. Serve Uvicorn asynchronously inside Colab's existing event loop
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

WARNING 09-15 22:56:57 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 09-15 22:56:57 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-15 22:56:57 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-15 22:56:57 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-15 22:56:57 [model.py:2021] Using max model len 1024
INFO 09-15 22:56:57 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=11954) INFO 09-15 22:57:01 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.f

(EngineCore pid=11954) Process EngineCore:
(EngineCore pid=11954) Traceback (most recent call last):
(EngineCore pid=11954)   File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
(EngineCore pid=11954)     self.run()
(EngineCore pid=11954)     ~~~~~~~~^^
(EngineCore pid=11954)   File "/usr/lib/python3.13/multiprocessing/process.py", line 108, in run
(EngineCore pid=11954)     self._target(*self._args, **self._kwargs)
(EngineCore pid=11954)     ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=11954)   File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core.py", line 1378, in run_engine_core
(EngineCore pid=11954)     raise e
(EngineCore pid=11954)   File "/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core.py", line 1336, in run_engine_core
(EngineCore pid=11954)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=11954)   File "/usr/local/lib/python3.13/dist-packages/vllm/tracing/otel.py", line 178

INFO 09-15 22:57:08 [utils.py:620] [shutdown] Process manager: send sigterm to process EngineCore


RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [3]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB

In [2]:
!pip uninstall -y torchaudio torchvision

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0


In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from contextlib import contextmanager

# 1. Patch sys.stdout / sys.stderr fileno for Jupyter environment
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

# 2. FIX: Override vLLM's internal suppress_stdout to prevent Colab fileno crashes
import vllm.utils.system_utils

@contextmanager
def dummy_suppress():
    yield

vllm.utils.system_utils.suppress_stdout = dummy_suppress

# Clean uninstall of torch and torchaudio to ensure consistent versions
!pip uninstall -y torch torchaudio

# Reinstall torch and torchaudio to match CUDA 13.0 (torch 2.13.0 is CUDA 13.0 compatible)
!pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu130
!pip install torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu130

# Set multiprocessing method for vLLM to 'fork' and explicitly set GPU memory utilization
# This ensures the worker process inherits the correct settings
os.environ["VLLM_GPU_MEMORY_UTILIZATION"] = "0.20"

print(f"Environment variable VLLM_GPU_MEMORY_UTILIZATION set to: {os.environ.get('VLLM_GPU_MEMORY_UTILIZATION')}")

# 3. Now import vLLM and FastAPI cleanly
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI()

# 4. Initialize vLLM Engine on Colab T4 GPU
engine_args = AsyncEngineArgs(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    gpu_memory_utilization=0.20,
    max_model_len=1024,
    enforce_eager=True  # Bypasses CUDA graph capture crashes on T4 GPUs
)

engine = AsyncLLMEngine.from_engine_args(engine_args)

class GenerateReq(BaseModel):
    prompt: str
    max_tokens: int = 128
    temperature: float = 0.7

@app.post("/remote-generate")
async def generate(req: GenerateReq):
    sampling_params = SamplingParams(
        temperature=req.temperature,
        max_tokens=req.max_tokens
    )
    request_id = f"colab-{asyncio.get_event_loop().time()}"
    results_generator = engine.generate(req.prompt, sampling_params, request_id)

    async def token_stream():
        previous_text = ""
        async for request_output in results_generator:
            text = request_output.outputs[0].text
            new_text = text[len(previous_text):]
            previous_text = text
            yield f"data: {new_text}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(token_stream(), media_type="text/event-stream")

# 5. Connect ngrok tunnel (Replace with your token from dashboard.ngrok.com)
NGROK_TOKEN = "3D0QjXb3z5iZRLU4NUe3KIqZai2_2YVGwrNybKRV3x8XWepcs"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8000)
print(f"🚀 Real GPU vLLM Server Live at: {public_url}/remote-generate")

# 6. Serve Uvicorn asynchronously inside Colab
config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

INFO 09-16 13:34:55 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-16 13:34:55 [model.py:2302] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 09-16 13:34:55 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-16 13:34:55 [model.py:2021] Using max model len 1024
INFO 09-16 13:34:55 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=2048.
WARNING 09-16 13:34:56 [vllm.py:1371] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-16 13:34:56 [vllm.py:1406] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 09-16 13:34:56 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

(EngineCore pid=8614) INFO 09-16 13:35:01 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_trace

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=8614) INFO 09-16 13:35:21 [default_loader.py:430] Loading weights took 1.58 seconds
(EngineCore pid=8614) INFO 09-16 13:35:22 [model_runner.py:404] Model loading took 0.93 GiB memory and 17.625537 seconds
(EngineCore pid=8614) WARNING 09-16 13:35:22 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=8614) INFO 09-16 13:35:22 [utils.py:306] Using LBNHC KV cache layout.
(EngineCore pid=8614) INFO 09-16 13:35:28 [gpu_worker.py:625] Available KV cache memory: 1.7 GiB
(EngineCore pid=8614) INFO 09-16 13:35:28 [kv_cache_utils.py:2032] GPU KV cache size: 148,864 tokens, Maximum concurrency for 1,024 tokens per request: 145.38x
(EngineCore pid=8614) INFO 09-16 13:35:29 [kernel_warmup.py:124] JIT kernel warmup starting.
(EngineCore pid=8614) INFO 09-16 13:35:29 [kernel_warmup.py:134] JIT kernel warmup finished in 0.00s.
(EngineCore pid=8614) INFO 09

INFO:     Started server process [8019]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     45.219.0.175:0 - "POST /remote-generate HTTP/1.1" 200 OK
WARNING 09-16 13:55:08 [input_processor.py:315] Passing raw prompts to InputProcessor is deprecated and will be removed in the future. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().
(EngineCore pid=8614) WARNING 09-16 13:55:08 [jit_monitor.py:141] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


In [ ]:
!pip uninstall -y torch torchaudio

# Reinstall torch and torchaudio to match CUDA 13.0 (torch 2.13.0 is CUDA 13.0 compatible)
!pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu130
!pip install torchaudio==2.11.0 --index-url https://download.pytorch.org/whl/cu130